# Training CTGAN and TVAE Models on UCI Adult Census Dataset

This notebook demonstrates training CTGAN and TVAE models using the demo adult dataset from the repository.

## Overview

**CTGAN** (Conditional Tabular GAN) and **TVAE** (Tabular Variational Autoencoder) are deep learning-based synthetic data generators for single table data.

### Paper Reference
*Lei Xu, Maria Skoularidou, Alfredo Cuesta-Infante, Kalyan Veeramachaneni.* **Modeling Tabular data using Conditional GAN**. NeurIPS, 2019.

This implementation uses the suggested training parameters from the paper.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from ctgan import CTGAN, TVAE, load_demo
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load the UCI Adult Census Dataset

The Adult Census dataset contains demographic information from the 1994 U.S. Census database. It's commonly used for classifying whether a person makes over $50K per year based on demographics.

Dataset source: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/2/adult)

In [2]:
# Load the demo adult dataset
print("Loading UCI Adult Census Dataset...")
real_data = load_demo()

print(f"✓ Dataset loaded successfully!")
print(f"  Rows: {real_data.shape[0]:,}")
print(f"  Columns: {real_data.shape[1]}")

Loading UCI Adult Census Dataset...
✓ Dataset loaded successfully!
  Rows: 32,561
  Columns: 15


##  3. Explore the Dataset

In [3]:
# Display column names
print("Dataset Columns:")
print(list(real_data.columns))
print("\n" + "="*60)

Dataset Columns:
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']



In [4]:
# Display first few rows
print("First 5 rows of the dataset:")
real_data.head()

First 5 rows of the dataset:


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [5]:
# Display dataset info
print("Dataset Information:")
real_data.info()

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [6]:
# Display summary statistics
print("Summary Statistics for Continuous Columns:")
real_data.describe()

Summary Statistics for Continuous Columns:


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


## 4. Define Discrete Columns

For CTGAN and TVAE to properly handle categorical data, we need to specify which columns are discrete (categorical).

In [7]:
# Define discrete (categorical) columns
discrete_columns = [
    'workclass',
    'education',
    'marital-status',
    'occupation',
    'relationship',
    'race',
    'sex',
    'native-country',
    'income'
]

print(f"Discrete columns ({len(discrete_columns)}): {discrete_columns}")
print(f"\nContinuous columns: {[col for col in real_data.columns if col not in discrete_columns]}")

Discrete columns (9): ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

Continuous columns: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']


## 5. Train CTGAN Model

**CTGAN** uses conditional generative adversarial networks with:
- Generator network that creates synthetic data
- Discriminator network that distinguishes real from synthetic data
- Conditional sampling to handle imbalanced categorical distributions

### Training Parameters (from paper)
- **epochs**: 300 (default, as recommended in the paper)
- **batch_size**: 500
- **generator_dim**: (256, 256)
- **discriminator_dim**: (256, 256)
- **pac**: 10 (packing samples to improve training)

In [8]:
print("="*60)
print("TRAINING CTGAN MODEL")
print("="*60)

# Initialize CTGAN with paper-suggested parameters
ctgan = CTGAN(
    epochs=300,           # As recommended in the paper
    batch_size=500,       # Default from paper
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    verbose=True
)

print("\nStarting CTGAN training...")
print("This may take some time (approximately 30-60 minutes depending on hardware)\n")

TRAINING CTGAN MODEL

Starting CTGAN training...
This may take some time (approximately 30-60 minutes depending on hardware)



In [9]:
# Train the model
ctgan.fit(real_data, discrete_columns)

print("\n✓ CTGAN training completed!")

Gen. (-00.28) | Discrim. (-00.07):  69%|██████▊   | 206/300 [3:06:44<1:25:12, 54.39s/it]  


KeyboardInterrupt: 

## 6. Train TVAE Model

**TVAE** uses a variational autoencoder architecture:
- Encoder that compresses data into a latent representation
- Decoder that reconstructs data from the latent space
- Generally faster to train than CTGAN

### Training Parameters
- **epochs**: 300 (as per repository defaults)
- **batch_size**: 500

In [ ]:
print("="*60)
print("TRAINING TVAE MODEL")
print("="*60)

# Initialize TVAE with default parameters
tvae = TVAE(
    epochs=300,           # Default parameter
    batch_size=500,
    verbose=True
)

print("\nStarting TVAE training...")
print("This typically trains faster than CTGAN\n")

In [ ]:
# Train the model
tvae.fit(real_data, discrete_columns)

print("\n✓ TVAE training completed!")

## 7. Generate Synthetic Data

Now that both models are trained, we can generate synthetic samples.

In [ ]:
print("Generating synthetic data...\n")

# Generate 1000 samples from CTGAN
print("[1/2] Generating 1000 samples from CTGAN...")
ctgan_synthetic_data = ctgan.sample(1000)
print(f"✓ CTGAN synthetic data shape: {ctgan_synthetic_data.shape}")

# Generate 1000 samples from TVAE
print("\n[2/2] Generating 1000 samples from TVAE...")
tvae_synthetic_data = tvae.sample(1000)
print(f"✓ TVAE synthetic data shape: {tvae_synthetic_data.shape}")

### Display Sample Synthetic Data

In [ ]:
print("CTGAN Synthetic Data - First 5 rows:")
ctgan_synthetic_data.head()

In [ ]:
print("TVAE Synthetic Data - First 5 rows:")
tvae_synthetic_data.head()

## 8. Save Trained Models

Save both models so they can be loaded and used later without retraining.

In [ ]:
print("Saving trained models...\n")

# Save CTGAN model
ctgan.save('ctgan_model.pkl')
print("✓ CTGAN model saved to: ctgan_model.pkl")

# Save TVAE model
tvae.save('tvae_model.pkl')
print("✓ TVAE model saved to: tvae_model.pkl")

# Save synthetic data
ctgan_synthetic_data.to_csv('ctgan_synthetic_data.csv', index=False)
print("✓ CTGAN synthetic data saved to: ctgan_synthetic_data.csv")

tvae_synthetic_data.to_csv('tvae_synthetic_data.csv', index=False)
print("✓ TVAE synthetic data saved to: tvae_synthetic_data.csv")

## 9. Load Saved Models

Demonstrate how to load the saved models for future use.

In [ ]:
print("Loading saved models...\n")

# Load CTGAN model
loaded_ctgan = CTGAN.load('ctgan_model.pkl')
print("✓ CTGAN model loaded successfully")

# Load TVAE model
loaded_tvae = TVAE.load('tvae_model.pkl')
print("✓ TVAE model loaded successfully")

print("\nYou can now use these models to generate more synthetic data!")

### Generate New Samples from Loaded Models

In [ ]:
# Generate 100 new samples from loaded CTGAN
new_ctgan_samples = loaded_ctgan.sample(100)
print(f"Generated {len(new_ctgan_samples)} new samples from loaded CTGAN model")
new_ctgan_samples.head()

In [ ]:
# Generate 100 new samples from loaded TVAE
new_tvae_samples = loaded_tvae.sample(100)
print(f"Generated {len(new_tvae_samples)} new samples from loaded TVAE model")
new_tvae_samples.head()

## 10. Summary

### Training Complete!

#### Models Trained:
1. **CTGAN** - Conditional Tabular GAN with 300 epochs
2. **TVAE** - Tabular Variational Autoencoder with 300 epochs

#### Files Generated:
- `ctgan_model.pkl` - Trained CTGAN model
- `tvae_model.pkl` - Trained TVAE model
- `ctgan_synthetic_data.csv` - 1000 synthetic samples from CTGAN
- `tvae_synthetic_data.csv` - 1000 synthetic samples from TVAE

#### Key Differences:
- **CTGAN**: Generally produces higher quality synthetic data but takes longer to train. Uses adversarial training.
- **TVAE**: Trains faster and can be more stable. Uses variational autoencoder architecture.

#### Next Steps:
1. Evaluate synthetic data quality using statistical measures
2. Compare distributions between real and synthetic data
3. Test privacy preservation
4. Use synthetic data for downstream tasks

---

**Citation:**
```
@inproceedings{ctgan,
  title={Modeling Tabular data using Conditional GAN},
  author={Xu, Lei and Skoularidou, Maria and Cuesta-Infante, Alfredo and Veeramachaneni, Kalyan},
  booktitle={Advances in Neural Information Processing Systems},
  year={2019}
}
```